### ライブラリのインストール


In [ ]:
!pip install --upgrade langchain langchain-openai langgraph pydantic python-dotenv faiss-cpu

In [ ]:
!pip install -U langchain-community

### API キーの取得


In [ ]:
from google.colab import userdata
import os

# サイドバーで追加したシークレットを取得
apikey = userdata.get("OPENAI_API_KEY")

# 改行や空白を除去して環境変数に登録
if apikey:
    os.environ["OPENAI_API_KEY"] = apikey.strip()
else:
    raise ValueError("ColabのSecretsに OPENAI_API_KEY が設定されていません")


### インポート


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

### LLM の設定


In [ ]:
# --- Step 5: ChatGPTモデルを準備 ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 温度0で安定回答


### おじさんたちの“迷言コーパス”を準備


In [ ]:
# --- Step 3: おじさんたちの“迷言コーパス”を準備 ---
docs = [
    "いかがなものか → 定義が不明確で、議論を凍結させる発言。",
    "持ち帰りましょうか → 決定を無限に延期する責任回避フレーズ。",
    "特段の問題はない → 課題を無視して空気で合意を装う言い回し。",
    "全会一致風ですね → HEL_AIが空気を学習しすぎたときの幻覚的ログ。"
]


### Embeddings 生成と FAISS ベクトルストア作成


In [ ]:
# --- Step 4: Embeddings生成とFAISSベクトルストア作成 ---
embeddings = OpenAIEmbeddings()  # OpenAIの埋め込みモデル
vectorstore = FAISS.from_texts(docs, embedding=embeddings)  # FAISSでベクトルDB構築


### RetrievalQA チェーンを作成


In [ ]:
# --- Step 6: RetrievalQAチェーンを作成 ---
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)


### 実行テスト


In [ ]:
# --- Step 7: おじさん1: いかがなものか ---
query = "……いかがなものか"
answer = qa.run(query)

print("🧓 いかがなものかおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 8: おじさん2: お持ち帰り ---
query = "では、いったん持ち帰りましょうか"
answer = qa.run(query)

print("👨‍💼 お持ち帰りおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 9: おじさん3: EQゼロ上司 ---
query = "まぁまぁ、特段の問題はないよね〜"
answer = qa.run(query)

print("👨‍💼 EQゼロ上司:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 10: おじさん4: HEL_AI自身のバグ ---
query = "全会一致風ですね"
answer = qa.run(query)

print("💀 HEL_AI(旧バグモード):", query)
print("🤖 HEL_AI(RAGモード):", answer)
